# Solar Cell Defect Classification with ResNet-18 (Two-Stage Fine-Tuning)

This code implements a **two-stage transfer learning pipeline** for classifying solar cells as `ok` or `defective`. It uses a **pretrained ResNet-18** and PyTorch.

## Key Features

- **Dataset Preparation**
  - Aggressive data augmentation for training: random resized crop, horizontal & vertical flip, brightness/contrast jitter.
  - Validation and test sets normalized only (no augmentation).

- **Data Splits**
  - 70% train / 15% validation / 15% test

- **Model**
  - Pretrained ResNet-18
  - Replace final layer with `Dropout + Linear(FC)` for 2 classes

- **Two-Stage Transfer Learning**
  1. Train only the final layer while freezing pretrained convolutional layers
  2. Unfreeze all layers and fine-tune the entire model

- **Training**
  - Loss: `CrossEntropyLoss`
  - Optimizer: Adam with weight decay
  - Epoch-wise accuracy printed during training

- **Evaluation**
  - Predict on test set
  - Generate **confusion matrix** and **classification report**
  - Use `softmax` to compute class probabilities

- **Advantages**
  - Better generalization due to data augmentation and staged fine-tuning
  - Clear test set evaluation for unbiased performance metrics

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
base_path = "/content/drive/MyDrive/SolarCellDataset/ResNetTrainData"

In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import Subset

# Load dataset twice (different transforms)
full_train_dataset = ImageFolder(base_path, transform=train_transforms)
full_valtest_dataset = ImageFolder(base_path, transform=val_test_transforms)

# Ensure same order
assert full_train_dataset.samples == full_valtest_dataset.samples

total_size = len(full_train_dataset)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

torch.manual_seed(42)

train_indices, val_indices, test_indices = random_split(
    range(total_size),
    [train_size, val_size, test_size]
)

train_dataset = Subset(full_train_dataset, train_indices.indices)
val_dataset = Subset(full_valtest_dataset, val_indices.indices)
test_dataset = Subset(full_valtest_dataset, test_indices.indices)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(pretrained=True)

# Replace final layer
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(num_features, 2)
)

model = model.to(device)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 204MB/s]


In [ ]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.fc.parameters(), lr=1e-3, weight_decay=1e-4)


In [ ]:
for epoch in range(20):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Calculate accuracy
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        # Print progress every 10 batches
        if batch_idx % 10 == 0:
            print(f"Epoch [{epoch+1}/20] Batch [{batch_idx}/{len(train_loader)}]")

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    print(f"Epoch [{epoch+1}/20] "
          f"Loss: {epoch_loss:.4f} "
          f"Accuracy: {epoch_acc:.2f}%\n")

Epoch [1/20] Batch [0/440]
Epoch [1/20] Batch [10/440]
Epoch [1/20] Batch [20/440]
Epoch [1/20] Batch [30/440]
Epoch [1/20] Batch [40/440]
Epoch [1/20] Batch [50/440]
Epoch [1/20] Batch [60/440]
Epoch [1/20] Batch [70/440]
Epoch [1/20] Batch [80/440]
Epoch [1/20] Batch [90/440]
Epoch [1/20] Batch [100/440]
Epoch [1/20] Batch [110/440]
Epoch [1/20] Batch [120/440]
Epoch [1/20] Batch [130/440]
Epoch [1/20] Batch [140/440]
Epoch [1/20] Batch [150/440]
Epoch [1/20] Batch [160/440]
Epoch [1/20] Batch [170/440]
Epoch [1/20] Batch [180/440]
Epoch [1/20] Batch [190/440]
Epoch [1/20] Batch [200/440]


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/SolarCellDataset/ResNetTrainData/Bgrade/061.jpg'

In [ ]:
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=1e-5)


In [ ]:
# Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

# New optimizer for all params
optimizer = optim.Adam(model.parameters(), lr=1e-5)

train_losses = []
val_losses = []
train_accs = []
val_accs = []

# Fine-tune loop (few epochs)
for epoch in range(15):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_losses.append(epoch_loss)
        train_accs.append(epoch_acc)
        val_losses.append(val_epoch_loss)
        val_accs.append(val_epoch_acc)

# Evaluation
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds))


# After training finishes
plt.figure(figsize=(12,5))

# Plot Loss
plt.subplot(1,2,1)
plt.plot(range(1,len(train_losses)+1), train_losses, label="Train Loss")
plt.plot(range(1,len(val_losses)+1), val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot Accuracy
plt.subplot(1,2,2)
plt.plot(range(1,len(train_accs)+1), train_accs, label="Train Accuracy")
plt.plot(range(1,len(val_accs)+1), val_accs, label="Val Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy Curve")
plt.legend()

plt.show()

UnidentifiedImageError: cannot identify image file <_io.BufferedReader name='/content/drive/MyDrive/SolarCellDataset/ResNetTrainData/Bgrade/061.jpg'>

In [ ]:
probs = torch.softmax(outputs, dim=1)
